In [ ]:
# ============================================================
# CELL 7: Stage 2 — Load instruction dataset
# ============================================================
# Before running: make sure instruction_dataset.jsonl is uploaded
# Format: {"instruction": "...", "response": "..."}  ← one JSON per line
# ============================================================

def load_instruction_dataset(jsonl_path: str) -> Dataset:
    """
    Loads the instruction-response Q&A pairs for Stage 2 SFT.

    The key difference from Stage 1:
    - Stage 1: raw text → model learns domain knowledge
    - Stage 2: structured Q&A → model learns HOW to answer questions

    We format each pair into the instruction prompt template:
    ### Instruction:
    <question>

    ### Response:
    <answer>

    This template tells the model where the question ends and
    the answer begins. Consistent format = consistent behaviour.
    """
    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(
            f"File not found: {jsonl_path}\n"
            "Upload instruction_dataset.jsonl to Colab."
        )

    # Load JSONL — one JSON object per line
    raw = load_dataset("json", data_files=jsonl_path, split="train")

    # Validate expected columns
    required = {"instruction", "response"}
    missing  = required - set(raw.column_names)
    if missing:
        raise ValueError(
            f"Missing columns: {missing}\n"
            f"Found: {raw.column_names}\n"
            "Your JSONL must have 'instruction' and 'response' keys."
        )

    def format_record(example):
        """
        Formats one Q&A pair into the prompt template.
        EOS token at the end tells the model where to stop generating —
        without this, the model never learns where a response should END,
        and it will keep hallucinating new fake questions after the real answer.
        """
        prompt = (
            f"### Instruction:\n{example['instruction'].strip()}\n\n"
            f"### Response:\n{example['response'].strip()}"
            f"{tokenizer.eos_token}"
        )
        return {"text": prompt}

    dataset = raw.map(format_record)

    print(f"✓ Instruction dataset loaded: {len(dataset)} pairs")
    print(f"  Columns: {dataset.column_names}")
    print(f"  Sample formatted text (first 300 chars):")
    print(f"  '{dataset[0]['text'][:300]}...'")

    return dataset


stage2_dataset = load_instruction_dataset(INSTRUCTION_PATH)
print(f"\n✓ Stage 2 dataset ready: {len(stage2_dataset)} training pairs")

In [ ]:
# ============================================================
# CELL 8: Stage 2 — Instruction Fine-Tuning (SFT)
# Expected time: 15-25 minutes on T4
# Expected final loss: ~0.8 to 1.4 (lower than Stage 1 — model is learning format)
# ============================================================
# This is the most important stage.
# This is where the visible quality jump happens.
# After this, the model answers placement questions specifically and correctly.
# ============================================================

print("=" * 55)
print("STAGE 2: INSTRUCTION FINE-TUNING (SFT)")
print("Goal: Teach the model to answer placement Q&A")
print(f"Starting from: Stage 1 merged model (already has domain knowledge)")
print("=" * 55)

# Load Stage 1 MERGED model — this is the key difference from the original notebook.
# We are NOT loading the base model again.
# We are loading a model that already has placement vocabulary from Stage 1.
# Stage 2 LoRA will now focus entirely on learning the Q&A response format
# rather than splitting effort between domain knowledge + format.
stage2_model, tokenizer = load_model_with_lora(STAGE1_MERGED)
FastLanguageModel.for_training(stage2_model)
tokenizer.padding_side = "right"  # right padding for SFT (standard)
stage2_dataset = load_instruction_dataset(INSTRUCTION_PATH)

stage2_config = SFTConfig(
    output_dir  = f"{OUTPUT_ROOT}/stage2_logs",

    num_train_epochs            = STAGE2_EPOCHS,
    # 3 epochs over 175 pairs = model sees each pair 3 times.
    # Enough for the instruction format to be well-learned.

    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,

    learning_rate  = STAGE2_LR,
    # 2e-4 — same as Stage 1 because our LoRA starts fresh here too.
    # We're training a brand new adapter on the Stage 1 merged model.

    warmup_steps   = WARMUP_STEPS,
    logging_steps  = LOGGING_STEPS,

    save_strategy  = "no",
    report_to      = "none",

    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    optim = "adamw_8bit",

    dataset_text_field = "text",
    max_length         = MAX_SEQ_LENGTH,

    # packing=False for instruction data — IMPORTANT difference from Stage 1.
    # Packing would mix different Q&A pairs into one sequence, which can
    # confuse the model about where one answer ends and the next question begins.
    # Each Q&A pair must be a complete, standalone training example.
    packing = False,

    seed = SEED,
)

stage2_trainer = SFTTrainer(
    model            = stage2_model,
    processing_class = tokenizer,
    train_dataset    = stage2_dataset,
    args             = stage2_config,
)

stage2_result = train_and_measure(stage2_trainer, "STAGE 2")

# ── Stage 2 Evaluation — run your showcase questions here ──────────────────
# This output goes directly into your reports/sft_model_comparison.md
# Copy these answers into your comparison table (base vs SFT column)
SHOWCASE_QUESTIONS = [
    "How should I prepare for Amazon SDE-1 placement?",
    "What DSA topics are most important for product company placements?",
    "I have a 6.5 CGPA. Can I still get into good companies?",
    "What is the TCS NQT exam pattern?",
    "How do I negotiate my first offer letter?",
]

print("\n" + "="*55)
print("STAGE 2 MODEL — SHOWCASE QUESTION ANSWERS")
print("Copy these into reports/sft_model_comparison.md")
print("="*55)
for q in SHOWCASE_QUESTIONS:
    print(f"\nQ: {q}")
    ans = generate_answer(stage2_model, tokenizer, q, max_new_tokens=200)
    print(f"A: {ans}")
    print("-" * 40)

# Save adapter + merge
save_and_merge(
    model      = stage2_model,
    tokenizer  = tokenizer,
    adapter_dir= STAGE2_ADAPTER,
    merged_dir = STAGE2_MERGED,
    stage_name = "Stage 2",
)

del stage2_trainer
del stage2_model
clear_gpu_memory()

print("\n✓ Stage 2 complete. Stage 2 merged model ready for Stage 3.")
print(f"  Merged model location: {STAGE2_MERGED}")